Lecture 13 will cover further go into examples on how to tie the GUI-widgets to the application logic by looking into more on the signal handling in Qt.

First a bit of standard stuff needed to work with Qt

In [ ]:
from PyQt5.QtCore import *
from PyQt5.QtGui import *
from PyQt5.QtWidgets import *
import sys
qt_app = QApplication(sys.argv)

# Signals

Lets start by creating a simpel model that emits a signal when the data has been modified.

In [ ]:
class MyCounterModel(QObject):
    # We create a new signal
    new_count = pyqtSignal()
    warning = pyqtSignal(str,)
    
    def __init__(self):
        super().__init__()
        self.count = 1

    def value(self):
        return self.count

    def increment(self, val=1):
        if val > 100:
            self.warning.emit('Input value to high!')
        else:
            self.count += val
            self.new_count.emit()


In [ ]:
class MyCounterView(QWidget):
    def __init__(self, model):
        super().__init__()
        self.button = QPushButton("Add 1")
        self.doubler = QPushButton("Double")
        self.label = QLabel()

        layout = QHBoxLayout()
        layout.addWidget(self.button)
        layout.addWidget(self.doubler)
        layout.addWidget(self.label)
        self.setLayout(layout)

        # Connect logic:
        # View part:
        self.model = model
        model.new_count.connect(self.update_value)
        model.warning.connect(self.display_warning)
        
        # Controller part:
        def add_one():
            model.increment()
        self.button.clicked.connect(add_one)
        
        def double_up():
            x = model.value()
            model.increment(x)       
        self.doubler.clicked.connect(double_up)
        
        self.update_value()  # Make sure it's up-to-date from the start.
    
    def update_value(self):
        self.label.setText("Value is " + str(self.model.value()))
        
    def display_warning(self, text):
        print(text)  # Display this to the user, somehow

In [ ]:
qt_app = QApplication.instance()

counter = MyCounterModel()
view = MyCounterView(counter)
view.show()

qt_app.exec_()

Lets start adressing the strange parts:
* The signal is declared as a static variable (class variable).
* The pyqtSignal() object doesn't actually have a connect or emit method.

It's all "magic" that the initialization of QObject takes care of. You don't need know how it happens, just how to use it.
*In order to use pyqtSignal()'s the class needs to inherit from QObject.*

# Creating reuseable Model and View components

Many games render the whole view using a single drawing area (not unlike the CardView class illustrated last lecture), and in these cases, there is really only 1 view that displays one 1 model, though for a more complex software it becomes completely unfeasable to keep track of all the state.

As such, it is typically better to create smaller Model's and View's that have a clear and simple purpose, and simply rely on those in the rest of the code.

Such components is typically simpler, with a clear purpose, and can be reuseable.

## Revisiting the AddOne game from lecture 12
We can revisit the previous game, but tackle the signals and model design with smaller models and views that take care of themselves:

In [ ]:
class PointModel(QObject):
    # This point model has 1 purpose: ensuring the signal is emitted whenever the value is changed
    new_value = pyqtSignal()

    def __init__(self):
        super().__init__()
        self.value = 0
    
    def add_one(self):
        self.value += 1
        self.new_value.emit()

    def clear(self):
        self.value = 0
        self.new_value.emit()


class Player:
    def __init__(self, name):
        self.name = name
        # The Player class does not need to concern itself with any signals
        # since the PointModel handles that itself!
        self.total = PointModel()


class AddOneGame(QObject):
    winner = pyqtSignal((str,))
    
    def __init__(self):
        super().__init__()
        self.players = [Player(name) for name in ['Micke', 'Kim', 'Thomas']]
    
    def player_click(self, index):
        # We must make sure not to directly modify total.value, else we won't let it emit its signal!
        # But the code in the "complex" AddOneGame is now much simpler;
        # No need to keep track of updating signals for all the values here, just pure game logic!
        self.players[index].total.add_one()
        self.check_winner()
    
    def reset(self):
        for player in self.players:
            player.total.clear()
    
    def check_winner(self):
        for player in self.players:
            if player.total.value >= 10:
                self.winner.emit(player.name + " won!")
                self.reset()
                break

**Emphasis**:
Note how the models above only contain state and logic. No graphical elements at all, no information on how it is displayed. It might just as well be played on the command line!

Below we will now introduce the View(s), which contain **no logic at all**. They make no assumptions on when the points are updated. They only decide how to display the state of the model to the user, and present the options for how to control the model.

In [ ]:
class PointView(QLabel):
    # The point view only needs to know about the PointModel and how to best display it, listening to the signal
    def __init__(self, point_model: PointModel):
        super().__init__()
        self.point_model = point_model
        self.point_model.new_value.connect(self.update_label)
        self.update_label()  # Initial refresh of textfield

    def update_label(self):
        self.setText(f'Points: {self.point_model.value}')


class PlayerView(QGroupBox):
    def __init__(self, player: Player, game: AddOneGame, player_number: int):
        super().__init__(player.name)

        # Connect button event to register in the game model:
        button = QPushButton('Add one')
        def click_event():
            return game.player_click(player_number)
        button.clicked.connect(click_event)

        # Set the layout of this player view:
        vbox = QVBoxLayout()
        vbox.addWidget(PointView(player.total))
        vbox.addWidget(button)
        self.setLayout(vbox)
    

class GameView(QWidget):
    def __init__(self, game_model):
        super().__init__()
        
        vbox = QVBoxLayout()
        for i, player in enumerate(game_model.players):
            vbox.addWidget(PlayerView(player, game, i))

        reset_button = QPushButton('Reset')
        reset_button.clicked.connect(game_model.reset)
        
        vbox.addWidget(reset_button)
        
        self.setLayout(vbox)
        self.show()

        # Connect the last signal:
        self.game = game_model
        game_model.winner.connect(self.alert_winner)

    def alert_winner(self, text: str):
        msg = QMessageBox()
        msg.setText(text)
        msg.exec()

In [ ]:
qt_app = QApplication.instance()
game = AddOneGame()
view = GameView(game)
qt_app.exec_()

# When to split off things to its own "sub-model"

This is a more subjective design question. Certainly, noone would split a complex number into a model for the real and imaginary parts respectively.

For example, strongly associated values

- Coordinate; longitud and latitude
- Brush shape in a drawing application; radius, sharpness
- Player money; how much money available and how much currently betted
- List of recent filenames

it seems logical that these could be their own class.

But what about When we have a collection of losely connected things, such as a "Player" model that contains the money and the cards on hand?
These often don't change at the same time, and have basically no overlap in functionality.
It is almost certainly neater, cleaner, and easier to reason about the code if these were kept separately.

Keeping separate signals also allows us to avoid unnecessary updating of variables that might not have changed; making for a snappier application

## TicTacToe

In [ ]:
class TTTPlayer(QObject):  # (tic tac toe player)
    wins_changed = pyqtSignal()
    
    def __init__(self, name):
        super().__init__()
        self.wins = 0
        self.name = name

    def win(self):
        self.wins += 1
        self.wins_changed.emit()


class TTTBoard(QObject):
    # there is only one thing this signal can refer to; the grid itself.
    # A slightly more generic name works fine here.
    grid_changed = pyqtSignal()

    @staticmethod
    def create_blank_grid(size):
        return [[None]*size for _ in range(size)]

    def __init__(self, size):
        super().__init__()
        self.size = size
        self.grid = self.create_blank_grid(self.size)

    def place(self, position, player_index):
        # Places a piece on the board from given player if valid.
        # Returns true is placement was valid.
        i, j = position
        if self.grid[i][j] is None:
            self.grid[i][j] = player_index
            self.grid_changed.emit()
            return True
        else:
            return False

    def clear(self):
        self.grid = self.create_blank_grid(self.size)
        self.grid_changed.emit()

    # A bit tricky logic to check for N number of values in a line
    def check_horizontal(self, length):
        for i in range(self.size):
            for j in range(self.size-length+1):
                value = self.grid[i][j]
                if value is not None:
                    for k in range(length):
                        if self.grid[i][j+k] != value:
                            break
                    else:
                        return value

    def check_vertical(self, length):
        for i in range(self.size-length+1):
            for j in range(self.size):
                value = self.grid[i][j]
                if value is not None:
                    for k in range(length):
                        if self.grid[i+k][j] != value:
                            break
                    else:
                        return value

    def check_leftdiag(self, length):
        for i in range(length-1, self.size):
            for j in range(self.size-length+1):
                value = self.grid[i][j]
                if value is not None:
                    for k in range(length):
                        if self.grid[i-k][j+k] != value:
                            break
                    else:
                        return value

    def check_rightdiag(self, length):
        for i in range(self.size-length+1):
            for j in range(self.size-length+1):
                value = self.grid[i][j]
                if value is not None:
                    for k in range(length):
                        if self.grid[i+k][j+k] != value:
                            break
                    else:
                        return value

    def check_winner(self, length):
        for check in [self.check_horizontal, self.check_vertical, self.check_leftdiag, self.check_rightdiag]:
            val = check(length)
            if val is not None:
                return val

    def is_full(self):
        for row in self.grid:
            for val in row:
                if val is None:
                    return False
        return True

                    
class TicTacToe(QObject):
    turn_changed = pyqtSignal()
    invalid_move = pyqtSignal() # No need to send a string if the reason is implied by the signal itself.
    winner = pyqtSignal(int)

    def __init__(self, names, board_size=3, win_length=3):
        super().__init__()
        self.players = [TTTPlayer(name) for name in names]
        self.turn = 0
        self.win_length = win_length
        self.board = TTTBoard(board_size)

    def next_turn(self):
        self.turn  = (self.turn + 1) % len(self.players)
        self.turn_changed.emit()

    def new_round(self):
        self.board.clear()  # Remember! We can't just re-create models since the view typicall has references to them.
        self.next_turn()  # Let the next player start next round.

    def place(self, position):
        if self.board.place(position, self.turn):
            # If valid, proceed:
            winner_num = self.board.check_winner(self.win_length)
            if winner_num is not None:
                self.players[winner_num].win()
                self.winner.emit(winner_num)
                self.new_round()
            elif self.board.is_full():
                self.new_round()
            else:
                self.next_turn()
        else:
            # We could also have possible use exceptions, or just return values to indicate failure
            self.invalid_move.emit()


Notice how we can technically play the game *without any graphics at all!*

In [ ]:
ttt = TicTacToe(['Micke', 'Kim', 'Thomas'], board_size=6, win_length=4)
# Fully useable as is:
# ttt.winner.connect(lambda x: print(f"{ttt.players[x].name} won"))
# ttt.place((1,1))
# ttt.place((2,1))
# ttt.place((0,2))

But lets create some nice views to display the entire game state!

In [ ]:
class PlayerView(QGroupBox):
    def __init__(self, num, player):
        super().__init__(f'{num}: {player.name}')
        self.player = player

        layout = QVBoxLayout()
        self.setLayout(layout)

        self.wins = QLabel()
        layout.addWidget(self.wins)

        player.wins_changed.connect(self.update_wins)
        self.update_wins()

    def update_wins(self):
        self.wins.setText("Wins: " + str(self.player.wins))

In [ ]:
class GridButton(QPushButton):
    def push(self):
        self.game.place(self.pos)
    
    def __init__(self, pos, game):
        super().__init__()
        self.pos = pos
        self.game = game
        self.clicked.connect(self.push)
        self.setMinimumWidth(5)


class TTTBoardView(QWidget):
    def __init__(self, game):
        super().__init__()
        layout = QGridLayout()
        self.setLayout(layout)
        self.buttons = []
        for i in range(game.board.size):
            self.buttons.append([])
            for j in range(game.board.size):
                button = GridButton((i, j), game)
                self.buttons[i].append(button)
                layout.addWidget(button, i, j)

        self.board_model = game.board
        self.board_model.grid_changed.connect(self.update_grid)
        self.update_grid()
        
    def update_grid(self):
        for i, row in enumerate(self.board_model.grid):
            self.buttons.append([])
            for j, value in enumerate(row):
                if value is None:
                    self.buttons[i][j].setText(' ')
                    self.buttons[i][j].setEnabled(True)
                else:
                    self.buttons[i][j].setText(f'{value}')
                    self.buttons[i][j].setEnabled(False)

In [ ]:
class TicTacToeView(QWidget):
    def __init__(self, game):
        super().__init__()

        self.game = game

        layout = QVBoxLayout()
        self.setLayout(layout)

        pbox = QHBoxLayout()
        layout.addLayout(pbox)

        for num, player in enumerate(game.players):
            pbox.addWidget(PlayerView(num, player))
        
        self.player_turn = QLabel()
        layout.addWidget(self.player_turn)
        self.game.turn_changed.connect(self.update_turn)

        layout.addWidget(TTTBoardView(game))

        game.winner.connect(self.present_winner)
        
        self.update_turn()

    def update_turn(self):
        self.player_turn.setText(f'{self.game.players[self.game.turn].name}s turn')

    def present_winner(self, player_number):
        box = QMessageBox()
        box.setText(f'{self.game.players[player_number].name} won')
        box.exec_()
        if self.game.players[player_number].wins >= 3:
            self.close()  # Enough gaming!


In [ ]:
qt_app = QApplication.instance()

view = TicTacToeView(ttt)
view.show()

qt_app.exec_()

## Tell or ask?

What is best?
* The Model sends a empty signal and the View(s) ask the Model for the information it needs.
* The Model sends the information in the signal

Probably no right answer, but there are some scenarions when passing a temporary state along might be the only reasonable option;
* Message/warning to the user
* The change of a value (e.g. how much it increased)
* Which index in a list model that was modified
